# コース分析：コーナー、ブレーキング＆加速ゾーン

このノートブックは、テレメトリーデータからコースの特徴を自動検出して可視化します。

## このノートブックの内容

- **コーナー検出**: GPS曲率データからコーナーを自動識別（方向（L/R）、エイペックス位置、概算半径を含む）
- **コーナーマップ**: 検出されたコーナーとエイペックスマーカーを表示するインタラクティブGPSマップ
- **ブレーキングゾーン検出**: 上位ラップで平均化した強いブレーキングの発生箇所を特定
- **加速ゾーン検出**: シフトチェンジのギャップを統合したスロットル適用ゾーンを特定
- **コースセグメント可視化**: ブレーキング（赤）、コーナー（オレンジ）、加速（緑）ゾーンを色分けしたGPSマップ

## 仕組み

1. **コーナー検出**: GPS座標からコースの曲率を計算し、持続する高曲率セクションを識別
2. **ゾーン平均化**: ベストラップタイムの103%以内のラップを分析し、一貫したブレーキング/加速パターンを検出
3. **セグメント作成**: コーナーとブレーキング/加速ゾーンを組み合わせて完全なコースセグメントを作成

## 自分のデータを使用する場合

自分のデータを分析するには：

1. 下の**最初のセルを実行**してパッケージをインストールし、アップロードウィジェットを表示
2. **「Choose File」をクリック**して`.xrk`、`.xrz`、または`.ibt`ファイルを選択
3. **検出パラメータの調整**（任意）: コーナー検出には調整可能なパラメータがあります：
   - `threshold=0.006`: 曲率しきい値（約167m半径）。低い値 = より緩やかなコーナーを検出
   - `min_corner_length=15`: サンプル数での最小コーナー長
   - `min_gap=80`: この距離（メートル）以内の同方向コーナーを統合
4. **残りのセルをすべて実行**してデータを分析

ステータスインジケーターに使用中のファイルが表示されます。ファイルをアップロードしない場合は、サンプルデータが使用されます。

## 必要なチャンネル

- GPSデータチャンネル（`GPS Latitude`、`GPS Longitude`、`GPS Speed`）
- ゾーン検出用のブレーキ圧（`BrakePress`）とスロットル（`PPS`）

**注意:** このノートブックはJupyterLite（ブラウザ）と通常のJupyterLab環境の両方で動作します。

In [ ]:
# 必要なパッケージをインストール（JupyterLiteで必要、通常のJupyterLabでは既にインストール済みならスキップ）
%pip install -q motorsports-data-notebook

# Rustパーサーバックエンドを使用（ファイル読み込みが約3倍高速）
import os

os.environ["LIBXRK_BACKEND"] = "rust"

# コアライブラリをインポート
import numpy as np
import plotly.graph_objects as go

# ヘルパー関数をインポート
from motorsports_data_notebook.channels import (
    get_best_lap_channels,
    get_top_laps,
)
from motorsports_data_notebook.corners import identify_corners
from motorsports_data_notebook.visualization import (
    format_lap_time,
    plot_track_segments,
    show_fig,
)
from motorsports_data_notebook.widgets import SessionPicker
from motorsports_data_notebook.zones import create_track_segments, detect_zones_averaged

# セッションピッカーとチャンネル設定
# 自分の.xrk/.xrz/.ibtファイルをアップロードするか、サンプルデータを使用
session = SessionPicker(
    default_file="../data/CMD_Inferno 86_Fuji GP Sh_Generic testing_a_2248.xrz",
    channel_mapping={
        # GPSチャンネル（コーナー検出に必要）
        "gps_latitude": "GPS Latitude",
        "gps_longitude": "GPS Longitude",
        "gps_speed": "GPS Speed",  # GPSからの速度（m/s）
        # ペダル入力（ゾーン検出に必要）
        "throttle": "PPS",  # スロットルポジションセンサー（0-100%）
        "brake": "BrakePress",  # ブレーキ圧（0-100%）
        # 車両ダイナミクス（スロットルアクセプタンス分析用、オプション）
        "lateral_g": "LateralAcc",  # 横G
        "steering": "SteerAngle",  # ステアリング角度（度）
    },
)
session.display()

In [ ]:
# 読み込んだセッションデータを取得
log = session.get_log()
laps = session.get_laps()
CHANNEL_NAMES = session.get_channel_names()

In [ ]:
# ラップタイム一覧を表示
laps.style.format({"lap_time": format_lap_time})  # type: ignore[dict-item]

In [ ]:
# libxrk 0.5.0のメソッドを使用してベストラップのチャンネルデータを抽出
gps_lat_ch = CHANNEL_NAMES["gps_latitude"]
gps_lon_ch = CHANNEL_NAMES["gps_longitude"]
best_lap, channels = get_best_lap_channels(
    log, laps, [gps_lat_ch, gps_lon_ch, "speed_kmh", "distance_m"]
)

# ベストラップでフィルタし、可視化用にGPS時間軸にリサンプル
best_lap_num = int(best_lap["num"])
aligned = (
    log.filter_by_lap(best_lap_num)
    .select_channels([gps_lat_ch, gps_lon_ch, "speed_kmh", "distance_m"])
    .resample_to_channel(gps_lat_ch)
    .channels
)

# コーナー検出と可視化用に配列に変換
lap_channels = {
    "GPS Latitude": aligned[gps_lat_ch].column(gps_lat_ch).to_numpy(),
    "GPS Longitude": aligned[gps_lon_ch].column(gps_lon_ch).to_numpy(),
    "speed_kmh": aligned["speed_kmh"].column("speed_kmh").to_numpy(),
    "distance_m": aligned["distance_m"].column("distance_m").to_numpy(),
}

In [ ]:
# GPS座標から直接コーナーを識別
# identify_cornersはGPS→XY変換、曲率計算、コーナー検出を処理
corners = identify_corners(
    lat=lap_channels["GPS Latitude"],
    lon=lap_channels["GPS Longitude"],
    threshold=0.003,  # 富士と袖ヶ浦のデータで調整済
    min_corner_length=15,  # 短いコーナーも検出するため小さめに設定
    min_gap=80,  # 80m以内の同方向コーナーを統合
)

print(f"{len(corners)}個のコーナーを検出:")
for c in corners:
    print(
        f"  {c.name} ({c.direction}): {c.start_dist:.0f}m - {c.end_dist:.0f}m (エイペックス {c.apex_dist:.0f}m, 半径 約{c.radius:.0f}m)"
    )

In [ ]:
# マーカー付きGPSマップ上にコーナーを可視化
fig = go.Figure()

# 速度で色分けしたコースをプロット
fig.add_trace(
    go.Scattermapbox(
        lat=lap_channels["GPS Latitude"],
        lon=lap_channels["GPS Longitude"],
        mode="markers",
        marker=dict(
            size=5,
            color=lap_channels["speed_kmh"],
            colorscale="Viridis",
            showscale=True,
            colorbar=dict(title="速度 (km/h)"),
        ),
        name="コース",
    )
)

# コーナーエイペックスマーカーを追加
for corner in corners:
    apex_idx = corner.apex_idx
    fig.add_trace(
        go.Scattermapbox(
            lat=[lap_channels["GPS Latitude"][apex_idx]],
            lon=[lap_channels["GPS Longitude"][apex_idx]],
            mode="markers+text",
            marker=dict(size=15, color="red"),
            text=[corner.name],
            textposition="top right",
            textfont=dict(size=12, color="red"),
            name=corner.name,
        )
    )

fig.update_layout(
    mapbox=dict(
        style="open-street-map",
        center=dict(
            lat=np.mean(lap_channels["GPS Latitude"]), lon=np.mean(lap_channels["GPS Longitude"])
        ),
        zoom=14,
    ),
    title="検出されたコーナー",
    showlegend=False,
    width=800,
    height=600,
)

show_fig(fig)

In [ ]:
# 上位ラップ全体で平均化したブレーキングゾーンと加速ゾーンを識別
top_laps = get_top_laps(laps, threshold_pct=1.03)

print(f"ベストラップタイム: {laps['lap_time'].min()}")
print(f"ゾーン平均化にベストの103%以内の{len(top_laps)}ラップを使用")

# 上位ラップ全体でゾーンを検出・平均化（ラップごとに必要なチャンネルのみを効率的に抽出）
braking_zones, accel_zones = detect_zones_averaged(log, top_laps, CHANNEL_NAMES)

In [ ]:
# コーナーとブレーキング/加速ゾーンを組み合わせた固定セグメント定義を作成
track_length = lap_channels["distance_m"][-1]
segments = create_track_segments(corners, braking_zones, accel_zones, track_length)

print(f"{len(segments)}個のコースセグメントを作成:")
for seg in segments:
    print(
        f"  [{seg.segment_type:12}] {seg.name:20} : {seg.start_dist:6.0f}m - {seg.end_dist:6.0f}m"
    )

In [ ]:
# GPSマップ上にコースセグメントを可視化
# plot_track_segments用にlap_channels辞書をDataFrameに変換
import pandas as pd

lap_channels_df = pd.DataFrame(lap_channels)

fig = plot_track_segments(
    lap_channels_df,
    segments,
    title="コースセグメント：ブレーキング（赤）、コーナー（オレンジ）、加速（緑）",
)
show_fig(fig)